# Racism Detection Pipeline — Loïc

**Objectif** : Entraîner un classifieur binaire (racist / not racist) avec BERTweet, évaluer le F1 par groupe ethnique ciblé, puis corriger le biais inter-groupes via une weighted loss.

**Pipeline** :
1. Chargement des données (`hatexplain.csv`)
2. Split train / val / test
3. Baseline SVM + TF-IDF
4. Fine-tuning BERTweet (loss standard)
5. Évaluation par groupe → mise en évidence du biais
6. Fine-tuning BERTweet (weighted loss par groupe)
7. Comparaison finale

## 0. Setup

In [ ]:
import os, sys, ast, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'Device: {DEVICE}')

MODEL_NAME = 'vinai/bertweet-base'
MAX_LEN    = 128
BATCH_SIZE = 16
LR         = 2e-5
EPOCHS     = 3
DATA_PATH  = '../data/final_datasets/hatexplain.csv'
GROUPS     = ['African', 'Asian', 'Jewish', 'Arab', 'Caucasian', 'Hispanic', 'Indian', 'Islam']

## 1. Chargement des données

In [ ]:
df = pd.read_csv(DATA_PATH)
df['targets_parsed'] = df['targets'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

print(f'Total samples : {len(df)}')
print(f'\nLabel distribution :')
print(df['label'].value_counts())

target_counts = Counter(
    t for targets in df[df['label']==1]['targets_parsed'] for t in targets if t != 'None'
)
print(f'\nGroupes ethniques ciblés (posts racistes) :')
print(pd.Series(target_counts).sort_values(ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Distribution des labels')
axes[0].set_xticklabels(['Not Racist (0)', 'Racist (1)'], rotation=0)
axes[0].set_ylabel('Count')
pd.Series(target_counts).sort_values(ascending=False).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Posts racistes par groupe ethnique ciblé')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 2. Split Train / Val / Test (70 / 15 / 15)

In [ ]:
df_train, df_temp = train_test_split(df,      test_size=0.30, random_state=SEED, stratify=df['label'])
df_val,   df_test = train_test_split(df_temp, test_size=0.50, random_state=SEED, stratify=df_temp['label'])
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)
print(f'Train : {len(df_train)} | Val : {len(df_val)} | Test : {len(df_test)}')
print(f'Train label dist : {df_train["label"].value_counts().to_dict()}')

## 3. Fonction d'évaluation par groupe
Pour chaque groupe G : F1 sur (posts racistes ciblant G) + (tous les posts label=0)

In [ ]:
def evaluate_per_group(df_eval, y_pred):
    results = {}
    y_pred = np.array(y_pred)
    for group in GROUPS:
        mask_pos = df_eval['targets_parsed'].apply(lambda t: group in t) & (df_eval['label'] == 1)
        mask_neg = df_eval['label'] == 0
        mask     = mask_pos | mask_neg
        if mask_pos.sum() < 5:
            continue
        results[group] = {
            'f1'   : f1_score(df_eval[mask]['label'].values, y_pred[mask], pos_label=1, zero_division=0),
            'n_pos': int(mask_pos.sum()),
        }
    return results

def print_group_results(results, model_name=''):
    print(f'\n--- F1 par groupe ({model_name}) ---')
    rows = [{'Groupe': g, 'F1': round(v['f1'], 3), 'N positifs': v['n_pos']}
            for g, v in sorted(results.items(), key=lambda x: -x[1]['f1'])]
    print(pd.DataFrame(rows).to_string(index=False))

## 4. Baseline — SVM + TF-IDF

In [ ]:
svm_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)),
    ('clf',   LinearSVC(C=1.0, max_iter=2000, random_state=SEED)),
])
svm_pipe.fit(df_train['text'], df_train['label'])
y_pred_svm    = svm_pipe.predict(df_test['text'])
svm_f1_global = f1_score(df_test['label'], y_pred_svm, average='macro')

print('=== SVM + TF-IDF ===')
print(classification_report(df_test['label'], y_pred_svm, target_names=['Not Racist', 'Racist']))
print(f'Macro F1 global : {svm_f1_global:.3f}')
svm_group_results = evaluate_per_group(df_test, y_pred_svm)
print_group_results(svm_group_results, 'SVM')

## 5. BERTweet — Dataset, DataLoader & fonctions d'entraînement

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class TweetDataset(Dataset):
    def __init__(self, texts, labels, sample_weights=None):
        self.texts          = list(texts)
        self.labels         = list(labels)
        self.sample_weights = sample_weights

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc  = tokenizer(self.texts[idx], max_length=MAX_LEN, padding='max_length',
                         truncation=True, return_tensors='pt')
        item = {'input_ids'     : enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'label'         : torch.tensor(self.labels[idx], dtype=torch.long)}
        if self.sample_weights is not None:
            item['weight'] = torch.tensor(self.sample_weights[idx], dtype=torch.float)
        return item

def make_loaders(df_tr, df_v, df_te, sample_weights=None):
    return (
        DataLoader(TweetDataset(df_tr['text'], df_tr['label'], sample_weights), batch_size=BATCH_SIZE, shuffle=True),
        DataLoader(TweetDataset(df_v['text'],  df_v['label']),                  batch_size=BATCH_SIZE),
        DataLoader(TweetDataset(df_te['text'], df_te['label']),                 batch_size=BATCH_SIZE),
    )

In [ ]:
def train_bertweet(train_loader, val_loader, use_sample_weights=False):
    model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer,
                    num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    ce_loss = nn.CrossEntropyLoss(reduction='none')
    history = {'train_loss': [], 'val_f1': []}

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(input_ids=batch['input_ids'].to(DEVICE),
                           attention_mask=batch['attention_mask'].to(DEVICE)).logits
            loss   = ce_loss(logits, batch['label'].to(DEVICE))
            if use_sample_weights and 'weight' in batch:
                loss = (loss * batch['weight'].to(DEVICE)).mean()
            else:
                loss = loss.mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in val_loader:
                logits = model(input_ids=batch['input_ids'].to(DEVICE),
                               attention_mask=batch['attention_mask'].to(DEVICE)).logits
                preds.extend(logits.argmax(-1).cpu().numpy())
                trues.extend(batch['label'].numpy())
        val_f1 = f1_score(trues, preds, average='macro')
        history['train_loss'].append(total_loss / len(train_loader))
        history['val_f1'].append(val_f1)
        print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Val Macro F1: {val_f1:.4f}')

    return model, history

def predict(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            logits = model(input_ids=batch['input_ids'].to(DEVICE),
                           attention_mask=batch['attention_mask'].to(DEVICE)).logits
            preds.extend(logits.argmax(-1).cpu().numpy())
    return np.array(preds)

## 6. Run 1 — BERTweet Baseline (loss standard)

In [ ]:
train_loader, val_loader, test_loader = make_loaders(df_train, df_val, df_test)
print('Entraînement BERTweet baseline...')
model_base, history_base = train_bertweet(train_loader, val_loader, use_sample_weights=False)

In [ ]:
y_pred_base   = predict(model_base, test_loader)
base_f1_global = f1_score(df_test['label'], y_pred_base, average='macro')

print('=== BERTweet Baseline ===')
print(classification_report(df_test['label'], y_pred_base, target_names=['Not Racist', 'Racist']))
print(f'Macro F1 global : {base_f1_global:.3f}')
base_group_results = evaluate_per_group(df_test, y_pred_base)
print_group_results(base_group_results, 'BERTweet Baseline')

## 7. Run 2 — BERTweet Weighted Loss par groupe
Les groupes sous-représentés reçoivent un poids plus élevé dans la loss.

In [ ]:
group_counts  = {g: max(1, df_train['targets_parsed'].apply(lambda t: g in t).sum()) for g in GROUPS}
max_count     = max(group_counts.values())
group_weights = {g: max_count / c for g, c in group_counts.items()}

print('Poids par groupe :')
for g, w in sorted(group_weights.items(), key=lambda x: -x[1]):
    print(f'  {g:<12} count={group_counts[g]:4d}  weight={w:.2f}')

def get_sample_weight(row):
    if row['label'] == 0:
        return 1.0
    targets = [t for t in row['targets_parsed'] if t in group_weights]
    return max((group_weights[t] for t in targets), default=1.0)

sample_weights = df_train.apply(get_sample_weight, axis=1).values
print(f'\nPoids — min: {sample_weights.min():.2f} | max: {sample_weights.max():.2f} | mean: {sample_weights.mean():.2f}')

In [ ]:
train_loader_w, val_loader_w, _ = make_loaders(df_train, df_val, df_test, sample_weights)
print('Entraînement BERTweet weighted...')
model_weighted, history_weighted = train_bertweet(train_loader_w, val_loader_w, use_sample_weights=True)

In [ ]:
y_pred_weighted    = predict(model_weighted, test_loader)
weighted_f1_global = f1_score(df_test['label'], y_pred_weighted, average='macro')

print('=== BERTweet Weighted Loss ===')
print(classification_report(df_test['label'], y_pred_weighted, target_names=['Not Racist', 'Racist']))
print(f'Macro F1 global : {weighted_f1_global:.3f}')
weighted_group_results = evaluate_per_group(df_test, y_pred_weighted)
print_group_results(weighted_group_results, 'BERTweet Weighted')

## 8. Comparaison Finale

In [ ]:
summary = pd.DataFrame({
    'Modèle'  : ['SVM + TF-IDF', 'BERTweet Baseline', 'BERTweet Weighted'],
    'Macro F1': [round(svm_f1_global, 3), round(base_f1_global, 3), round(weighted_f1_global, 3)],
})
print('=== Résultats globaux ===')
print(summary.to_string(index=False))

In [ ]:
all_groups = sorted(set(base_group_results) & set(weighted_group_results) & set(svm_group_results))
group_df   = pd.DataFrame({
    'Groupe'            : all_groups,
    'N positifs'        : [base_group_results[g]['n_pos'] for g in all_groups],
    'SVM F1'            : [round(svm_group_results[g]['f1'], 3) for g in all_groups],
    'BERTweet Base F1'  : [round(base_group_results[g]['f1'], 3) for g in all_groups],
    'BERTweet Weight F1': [round(weighted_group_results[g]['f1'], 3) for g in all_groups],
})
print('=== F1 par groupe ethnique ===')
print(group_df.to_string(index=False))

In [ ]:
x, width = np.arange(len(all_groups)), 0.25
fig, ax  = plt.subplots(figsize=(12, 5))
ax.bar(x - width, [svm_group_results[g]['f1']      for g in all_groups], width, label='SVM',              color='steelblue', alpha=0.85)
ax.bar(x,         [base_group_results[g]['f1']      for g in all_groups], width, label='BERTweet Baseline', color='tomato',    alpha=0.85)
ax.bar(x + width, [weighted_group_results[g]['f1']  for g in all_groups], width, label='BERTweet Weighted', color='seagreen',  alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(all_groups, rotation=30, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('F1 par groupe ethnique ciblé — Comparaison des modèles')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('../data/final_datasets/group_f1_comparison.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_base['train_loss'],     label='Baseline', color='tomato')
axes[0].plot(history_weighted['train_loss'], label='Weighted', color='seagreen')
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].plot(history_base['val_f1'],     label='Baseline', color='tomato')
axes[1].plot(history_weighted['val_f1'], label='Weighted', color='seagreen')
axes[1].set_title('Validation Macro F1')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()